#### **Bioconductor -- FETCH**

In [1]:
#!pip install requests beautifulsoup4

In [2]:
import re, time, json, csv, html
from pathlib import Path
from urllib.parse import urlencode
from functools import lru_cache
import requests
from bs4 import BeautifulSoup
from collections import defaultdict
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import random
from typing import Union
import pandas as pd

In [3]:
# ========================
# Paths & output
# ========================
TAGS_PATH  = Path("../02-datasource/qiime2_tags_[filtered].json")
OUT_DIR    = Path("../04-output/raw/qiime2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

JSONL_PATH = OUT_DIR / "qiime2_metagenomics_tag.jsonl"
CSV_PATH   = OUT_DIR / "qiime2_metagenomics_tag.csv"
CACHE_IDS  = OUT_DIR / "seen_ids_tag.txt"

HEADERS = {
    "User-Agent": "MetagenomicsCollector/1.3 (+research; contact: genereux.akotenou@gmail.com)"
}

# ========================
# Crawl config
# ========================
SLEEP_BETWEEN_REQUESTS = 1.1
ORDER_VARIANTS = ["posts", "views", "activity"]


In [4]:
# ========================
# Small helpers
# ========================
def sleep():
    # time.sleep(SLEEP_BETWEEN_REQUESTS)
    time.sleep(SLEEP_BETWEEN_REQUESTS + random.uniform(0, 0.6))

SESSION = requests.Session()
retry = Retry(
    total=8,                # overall retries
    connect=4,              # connection-level retries
    read=4,                 # read timeouts
    backoff_factor=1.2,     # exponential backoff: 1.2, 2.4, 4.8, ...
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset(["GET"]),
    raise_on_status=False,
)
adapter = HTTPAdapter(max_retries=retry, pool_connections=100, pool_maxsize=100)
SESSION.mount("https://", adapter)
SESSION.mount("http://", adapter)

def load_seen_ids():
    if CACHE_IDS.exists():
        return set(x.strip() for x in CACHE_IDS.read_text().splitlines() if x.strip())
    return set()

def save_seen_ids(ids):
    with open(CACHE_IDS, "w", encoding="utf-8") as f:
        for pid in sorted(ids, key=lambda x: int(x)):
            f.write(f"{pid}\n")

def fetch(url, as_json=False, params=None, timeout=(5, 60)):
    """
    Robust fetch with retries/backoff. timeout=(connect, read)
    """
    for attempt in range(1, 6):
        try:
            r = SESSION.get(url, headers=HEADERS, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json() if as_json else r.text
        except requests.exceptions.RequestException as e:
            if attempt == 5:
                # give up after 5 explicit tries (Retry may have already retried inside)
                raise
            delay = min(15, (2 ** attempt) * 0.8) + random.uniform(0, 0.5)
            print(f"[retry] {type(e).__name__}: {e} — sleeping {delay:.1f}s then retrying")
            time.sleep(delay)

def html_plain_text(xhtml):
    try:
        return BeautifulSoup(xhtml or "", "html.parser").get_text("\n", strip=True)
    except Exception:
        return (xhtml or "").strip()

def as_int_or_str(x):
    try:
        return int(x)
    except Exception:
        return str(x)

def normalize_tags(tag_val, fallback=None):
    # API returns space-separated string; sometimes list
    if isinstance(tag_val, list):
        return [t for t in tag_val if t]
    if isinstance(tag_val, str):
        toks = [t.strip() for t in tag_val.split() if t.strip()]
        if toks:
            return toks
    return list(fallback) if fallback else []

def get_body_text(post_json: dict) -> str:
    # Biostars uses 'xhtml' or 'content'
    html_blob = post_json.get("xhtml") or post_json.get("content") or ""
    return html_plain_text(html_blob)

# Keep everything in tag-only mode
def apply_labels_from_config(_text): return []
def blocked_by_global_excludes(_text): return False

# ========================
# Read list of tags
# ========================
def load_tags_list(path: Path):
    if path.exists():
        try:
            tags = json.loads(path.read_text(encoding="utf-8"))
            tags = [t.strip() for t in tags if isinstance(t, str) and t.strip()]
            if tags:
                return tags
        except Exception as e:
            print(f"[warn] failed to read {path}: {e}")
    # fallback to your example list
    return ["rna-seq", "metagenomics", "microbiome", "shotgun-metagenomics", "16s", "assembly", "binning"]

TAGS = load_tags_list(TAGS_PATH)
print(f"Loaded {len(TAGS)} tags.")

# ========================
# Tag URLs (single page per order)
# ========================
def tag_order_urls(tag):
    base = f"https://forum.qiime2.org/tag/{tag}/l/latest?ascending=false&period=all"
    for order in ORDER_VARIANTS:
        yield f"{base}&order={order}", order

def extract_post_ids_from_index(html_text):
    soup = BeautifulSoup(html_text, "html.parser")
    ids = set()

    for a in soup.select("a.raw-topic-link"):
        href = a.get("href", "")
        m = re.search(r"/t/[^/]+/(\d+)", href)
        if m:
            ids.add(m.group(1))

    return ids

Loaded 33 tags.


In [5]:
# ========================
# API helpers
# ========================
@lru_cache(maxsize=8192)
def get_post_json(pid):
    return fetch(f"https://support.bioconductor.org/api/post/{pid}/", as_json=True)

@lru_cache(maxsize=4096)
def get_user_json(uid):
    """Fetch Biostars user profile once; cached."""
    if uid in (None, ""):
        return {}
    try:
        return fetch(f"https://support.bioconductor.org/api/user/{uid}/", as_json=True)
    except Exception:
        return {}

def compact_user_profile(u):
    if not u:
        return None
    return {
        "id": u.get("id"),
        "name": u.get("name"),
        "date_joined": u.get("date_joined"),
        "last_login": u.get("last_login"),
        "joined_days_ago": u.get("joined_days_ago"),
        # votes GIVEN by the user
        "vote_count": u.get("vote_count"),
    }

def make_author_obj(name, uid):
    """Return merged author object with profile fields (uid may be str)."""
    if uid in (None, ""):
        return {
            "id": None, "name": name, "date_joined": None,
            "last_login": None, "vote_count": None, "joined_days_ago": None
        }
    uid_norm = as_int_or_str(uid)
    prof_raw = compact_user_profile(get_user_json(uid_norm)) or {}
    return {
        "id": uid_norm,
        "name": name,
        "date_joined": prof_raw.get("date_joined"),
        "last_login": prof_raw.get("last_login"),
        "joined_days_ago": prof_raw.get("joined_days_ago"),
        "vote_count": prof_raw.get("vote_count"),
    }

# ========================
# HTML: enumerate answers & accepted
# ========================
def extract_answer_ids_and_accept_flag(soup):
    """
    Return (answer_ids, accepted_id) from a Biostars thread HTML.
    - Each answer is under: div.post.view.open[data-value]
    - Accepted answer body has itemprop="acceptedAnswer"
    - Other answers use itemprop="suggestedAnswer"
    """
    ans_ids, accepted_id = [], None
    for block in soup.select('div.post.view.open[data-value]'):
        pid = block.get("data-value")
        if not pid:
            continue
        body = block.select_one(
            'div.body[itemprop="acceptedAnswer"], div.body[itemprop="suggestedAnswer"]'
        )
        if body:
            ans_ids.append(pid)
            if body.get("itemprop") == "acceptedAnswer":
                accepted_id = pid
    return ans_ids, accepted_id

# ========================
# Thread parsing (API + HTML) incl. user profiles
# ========================
def parse_thread_page(tid):
    """
    Fetch a Qiime2 Forum thread using the official Discourse JSON API.
    """
    url = f"https://forum.qiime2.org/t/{tid}.json"
    data = fetch(url, as_json=True)

    # Basic metadata
    title = data.get("title", "")
    tags  = data.get("tags", [])
    posts = data.get("post_stream", {}).get("posts", [])

    parsed_posts = []
    for p in posts:
        cooked_html = p.get("cooked", "") or ""
        text = BeautifulSoup(cooked_html, "html.parser").get_text("\n", strip=True)

        # Extract likes: actions_summary ID=2 = like
        like_entry = next((a for a in p.get("actions_summary", []) if a.get("id") == 2), None)
        like_count = like_entry.get("count", 0) if like_entry else 0

        parsed_posts.append({
            "post_id": p.get("id"),
            "post_number": p.get("post_number"),
            "user_id": p.get("user_id"),
            "username": p.get("username"),
            "created_at": p.get("created_at"),
            "text": text,
            "likes": like_count,
            "reply_to": p.get("reply_to_post_number"),
        })

    if not parsed_posts:
        return {
            "url": url,
            "title": title,
            "tags": tags,
            "question": None,
            "answers": []
        }

    question = parsed_posts[0]
    answers  = parsed_posts[1:]

    return {
        "url": url,
        "title": title,
        "tags": tags,
        "question": question,
        "answers": answers
    }



# ========================
# Normalizer (adds answers_all, origin tag + order)
# ========================
def normalize_record(pid, thread_meta, origin_tag="", order_variant=""):
    question = thread_meta.get("question")
    if not question or not question.get("text"):
        return None

    q_text = question["text"].strip()
    answers = thread_meta.get("answers", [])

    # ---------------------------------------------------------
    # Build single block of answers: ANSWER1, ANSWER2, ...
    # ---------------------------------------------------------
    answer_lines = []
    for i, ans in enumerate(answers, start=1):
        txt   = ans.get("text", "").strip()
        likes = ans.get("likes", 0)

        line = f"ANSWER {i}: {txt} ({likes} LIKE{'S' if likes != 1 else ''})"
        answer_lines.append(line)

    # Join them with separators ("---")
    answers_text_block = "\n---\n".join(answer_lines)
    # ---------------------------------------------------------

    return {
        "source": "qiime2",
        "topic_id": pid,
        "url": thread_meta.get("url"),
        "title": thread_meta.get("title"),

        "search_query": origin_tag,
        "crawl_tag": origin_tag,
        "crawl_order": order_variant,

        "question": q_text,
        "question_author": question.get("username"),
        "question_created_at": question.get("created_at"),

        "answers": answers,
        "answer_count": len(answers),
        "tags": thread_meta.get("tags", []),

        # NEW FIELD
        "best_answer": answers_text_block,
    }
    
def preview_csv(csv_path: Union[str, Path], name: str = "bioconductor QA Preview"):
    p = Path(csv_path)
    if not p.exists():
        raise FileNotFoundError(f"CSV not found: {p}")
    df = pd.read_csv(p)
    return df


In [12]:
# ========================
# Crawl (by tag, one page per order)
# ========================
from collections import defaultdict

seen = load_seen_ids()
collected_total = 0
drop_stats = defaultdict(int)

with open(JSONL_PATH, "a", encoding="utf-8") as jf:
    for tag in TAGS:
        print(f"\n=== TAG: {tag} ===")
        term_collected = 0
        for url, order_name in tag_order_urls(tag):
            try:
                html_text = fetch(url)
            except Exception as e:
                print("Tag page fetch failed:", url, e)
                sleep(); continue
            sleep()

            pids = extract_post_ids_from_index(html_text)
            if not pids:
                print(f"[{tag}|{order_name}] no post ids on {url}")
                continue

            print(f"[{tag}|{order_name}] {len(pids)} posts on {url}")

            for pid in sorted(pids, key=lambda x: int(x)):
                if pid in seen:
                    continue
                try:
                    thread_meta = parse_thread_page(pid); sleep()
                    # print(thread_meta)
                    post_json   = get_post_json(pid);     sleep()

                    rec = normalize_record(pid, thread_meta, origin_tag=tag, order_variant=order_name)
                    if rec:
                        jf.write(json.dumps(rec, ensure_ascii=False) + "\n")
                        jf.flush()
                        collected_total += 1
                        term_collected += 1
                        seen.add(pid)

                        if collected_total % 25 == 0:
                            print(f"Collected {collected_total} items so far...")
                    else:
                        if not (thread_meta.get("question_text") or "").strip():
                            drop_stats["empty_question"] += 1
                        elif not (thread_meta.get("best_answer") or {}).get("text"):
                            drop_stats["no_answer"] += 1
                        else:
                            drop_stats["excluded_or_other"] += 1
                except requests.HTTPError as e:
                    print(f"[warn] HTTP error pid={pid}: {e}")
                    sleep()
                except Exception as e:
                    print(f"[warn] failed pid={pid}: {e}")
                    sleep()

            save_seen_ids(seen)
            sleep()

        print(f"[{tag}] collected={term_collected}")

print(f"\nDone. Collected {collected_total} items → {JSONL_PATH}")
print("Drop stats:", dict(drop_stats))


=== TAG: metagenomics ===
[metagenomics|posts] 12 posts on https://forum.qiime2.org/tag/metagenomics/l/latest?ascending=false&period=all&order=posts
[metagenomics|views] 12 posts on https://forum.qiime2.org/tag/metagenomics/l/latest?ascending=false&period=all&order=views
[metagenomics|activity] 12 posts on https://forum.qiime2.org/tag/metagenomics/l/latest?ascending=false&period=all&order=activity
[metagenomics] collected=12

=== TAG: wgs ===
[wgs|posts] 2 posts on https://forum.qiime2.org/tag/wgs/l/latest?ascending=false&period=all&order=posts
[wgs|views] 2 posts on https://forum.qiime2.org/tag/wgs/l/latest?ascending=false&period=all&order=views
[wgs|activity] 2 posts on https://forum.qiime2.org/tag/wgs/l/latest?ascending=false&period=all&order=activity
[wgs] collected=2

=== TAG: virome ===
[virome|posts] 2 posts on https://forum.qiime2.org/tag/virome/l/latest?ascending=false&period=all&order=posts
[virome|views] 2 posts on https://forum.qiime2.org/tag/virome/l/latest?ascending=fals

In [6]:
INPUT_JSONL = JSONL_PATH
OUTPUT_CSV  = "/home/biolab-office-1/DATALAB/2025/Genomeer/dataset/04-output/hq/qiime2/qiime2_db.csv"

with open(INPUT_JSONL, "r", encoding="utf-8") as jf, \
     open(OUTPUT_CSV, "w", encoding="utf-8", newline="") as cf:

    writer = csv.writer(cf)
    writer.writerow(["question", "answer", "tags", "accepted", "score", "url"])

    for line in jf:
        rec = json.loads(line)

        question = rec.get("question", "").replace("\n", " ").strip()
        answer   = rec.get("best_answer", "").replace("\n", " ").strip()

        # tags comma-separated
        tags = rec.get("tags", [])
        tags_str = ",".join(tags)

        # accepted column (empty for Qiime2 forum)
        accepted = ""

        # score = total likes / number of answers
        answers = rec.get("answers", [])
        if answers:
            # total_likes = sum(a.get("likes", 0) for a in answers)
            # score = total_likes / len(answers)
            score = max([a.get("likes", 0) for a in answers])
        else:
            score = 0

        url = rec.get("url", "")

        writer.writerow([question, answer, tags_str, accepted, score, url])


In [7]:
df = preview_csv("/home/biolab-office-1/DATALAB/2025/Genomeer/dataset/04-output/hq/qiime2/qiime2_db.csv")
df.head()

,question,answer,tags,accepted,score,url
0,We are proud to announce beta release of the q...,"ANSWER 1: Very exciting, thanks for all of you...","taxonomy,metagenomics",NaN,4,https://forum.qiime2.org/t/25296.json
1,can anyone guide me from where i can get pre-f...,"ANSWER 1: Hi @Jaykishan_Solanki , welcome to :...","taxonomy,feature-classifier,qiime2r,metagenomi...",NaN,1,https://forum.qiime2.org/t/27285.json
2,Dear Qiimers! I am happy to play with a large ...,"ANSWER 1: Hi @timanix , I cant speak to others...",metagenomics,NaN,4,https://forum.qiime2.org/t/28516.json
3,Hello. I always thanks for your help. I want t...,"ANSWER 1: Hello @sooni , Could you explain wha...",metagenomics,NaN,1,https://forum.qiime2.org/t/28963.json
4,Hello Qiime2 Community! I'm excited to introdu...,NaN,"plugin-development,metagenomics",NaN,0,https://forum.qiime2.org/t/29007.json


In [8]:
df.shape

(402, 6)